<!-- notebook-header -->
# Series Temporais: Fundamentos

**Modulo:** 05 - Dominios Aplicados / 05D - Series Temporais  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Componentes temporais, estacionariedade, ACF, PACF, diferenciacao e suavizacao.


# Series Temporais: Fundamentos## Analise de Dados Sequenciais no Tempo**Objetivo**: Compreender conceitos basicos de series temporais, componentes, transformacoes e deteccao de padroes.**Contexto**: Series temporais aparecem em economia, meteorologia, saude, finanzas e IoT. Sao dados onde o tempo eh uma dimensao critica.

## 1. O que sao Series Temporais?Uma **serie temporal** eh uma sequencia de observacoes coletadas em instantes regulares (ou irregulares) de tempo.### Analogias Intuitivas:- **Batimento cardiaco**: pulsos registrados ao longo do tempo. Padrao regular com variacoes.- **Preco de acoes**: cotacoes que fluem, com tendencias e picos.- **Temperatura diaria**: ciclos sazonais (invierno mais frio, verao mais quente).- **Vendas mensais**: crescimento (tendencia) + picos em datas especiais (sazonalidade) + aleatoriedade (ruido).### O que observar:1. A natureza dependente do tempo dos dados2. A importancia da ordem cronologica3. Padroes que repetem (sazonalidade)4. Mudancas graduais (tendencias)5. Flutuacoes aleatorias sobrepostas### O que concluir:1. Series temporais nao sao dados independentes2. Metodos ML convencionais podem falhar3. Tempo eh dimensao fundamental4. Dependencia serial invalida hipoteses i.i.d.5. Ferramentas especializadas sao necessarias### Diferenca Fundamental:Em dados convencionais (ML padrao), a **ordem das linhas nao importa**. Em series temporais, a **ordem eh tudo**. O valor em t depende frequentemente de t-1, t-2, etc.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Serie temporal simples: 100 dias
np.random.seed(42)
dias = np.arange(100)

# Preco de acao simulado: tendencia + sazonalidade + ruido
tendencia = dias * 0.5 + 50
sazonalidade = 5 * np.sin(2 * np.pi * dias / 30)
ruido = np.random.normal(0, 2, 100)
preco = tendencia + sazonalidade + ruido

plt.figure(figsize=(12, 4))
plt.plot(dias, preco, linewidth=2)
plt.xlabel("Dia")
plt.ylabel("Preco")
plt.title("Serie Temporal: Preco de Acao (100 dias)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/serie_temporal_simples.png", dpi=100, bbox_inches="tight")
plt.show()

print("Observacoes:")
print(f"Preco inicial: {preco[0]:.2f}")
print(f"Preco final: {preco[-1]:.2f}")
print(f"Minimo: {np.min(preco):.2f}")
print(f"Maximo: {np.max(preco):.2f}")


### O que observar:1. A serie tem uma tendencia de crescimento2. Flutuacoes regulares (sazonalidade) aparecem a cada ~30 dias3. Ruido aleatorio (variabilidade nao explicada) esta presente4. O valor em cada dia eh dependente dos dias anteriores### O que concluir:1. Series temporais sao compostas por multiplos componentes2. Visualizacao eh crucial para entender a estrutura3. Dependencia temporal invalida muitos metodos ML clasicos4. Precisamos de ferramentas especializadas para series temporais

## 2. Componentes de Series TemporaisToda serie temporal pode ser decomposta em:1. **Tendencia (Trend)**: direcao geral da serie a longo prazo (crescimento/queda)2. **Sazonalidade (Seasonality)**: padroes repetitivos em periodos fixos (diario, mensal, anual)3. **Ciclo**: flutuacoes longas (> 1 ano) nao periodicas estritamente4. **Ruido (Noise)**: variabilidade aleatoria irredutivel### Modelos de Decomposicao:- **Aditivo**: Y = Tendencia + Sazonalidade + Ruido- **Multiplicativo**: Y = Tendencia × Sazonalidade × RuidoEscolha aditivo quando componentes sao independentes (amplitudes fixas).Escolha multiplicativo quando sazonalidade cresce com tendencia.

In [ ]:
# Decomposicao aditiva manualmente
def decompose_aditiva(y, periodo):
    n = len(y)
    # Tendencia: media movel centrada
    tendencia = np.zeros(n)
    janela = periodo
    for i in range(n):
        inicio = max(0, i - janela // 2)
        fim = min(n, i + janela // 2 + 1)
        tendencia[i] = np.mean(y[inicio:fim])
    # Sazonalidade: media por posicao no ciclo
    y_detrended = y - tendencia
    sazonalidade = np.zeros(n)
    for fase in range(periodo):
        indices = np.arange(fase, n, periodo)
        if len(indices) > 0:
            sazonalidade[indices] = np.mean(y_detrended[indices])
    # Ruido: residuo
    ruido = y - tendencia - sazonalidade
    return tendencia, sazonalidade, ruido


### O que observar:1. Cada componente tem uma "escala" diferente2. Tendencia eh lisa (smooth), sazonalidade eh periodica, ruido eh aleatorio3. Somando os tres componentes recuperamos a serie original4. O ruido tem media aproximadamente zero### O que concluir:1. Decomposicao eh valiosa para entender a estrutura da serie2. Cada componente pode ser analisado separadamente3. Reducao de ruido (filtragem) eh importante antes de prever tendencias4. Periodicidade pode ser estimada visualmente

## 3. EstacionariedadeUma serie eh **estacionaria** se suas propriedades estatisticas (media, variancia, autocorrelacao) nao mudam ao longo do tempo.### Propriedades:- **Media constante**: E[Y_t] = mu (independente de t)- **Variancia constante**: Var(Y_t) = sigma^2 (independente de t)- **Autocorrelacao constante**: dependencia entre Y_t e Y_{t-k} nao muda com t### Por que importa?- Series estacionarias sao mais faceis de modelar- Modelos ARIMA funcionam bem com series estacionarias- Predicoes sao mais confiaveis em regimes estacionarios### Teste: ADF SimplificadoUma aproximacao simples: calcular a correlacao entre Y_t e Y_{t-1}. Se eh alta, serie nao eh estacionaria.

In [ ]:
def teste_estacionariedade_simples(y, lag=1):
    # Correlacao entre y[t] e y[t-lag]
    y_atual = y[lag:]
    y_passada = y[:-lag]
    if len(y_atual) == 0:
        return float("nan")
    correlacao = np.corrcoef(y_atual, y_passada)[0, 1]
    return correlacao


### O que observar:1. Correlacao alta (>0.8) indica nao-estacionariedade2. Series com tendencia clara tem correlacao proxima a 13. Ruido branco tem correlacao proxima a 04. Estacionariedade eh visual mas confirmada por testes numericos### O que concluir:1. Tendencia causa nao-estacionariedade2. Sazonalidade tambem causa nao-estacionariedade3. Estacionariedade eh requisito para muitos metodos4. Precisamos transformar series nao-estacionarias

## 4. Diferenciacao para Estacionariedade**Diferenciacao**: Y'_t = Y_t - Y_{t-1}Diferenciacao remove tendencias lineares. Diferenciacao dupla remove tendencias quadraticas.### Ideia:Se a serie tem tendencia linear, suas primeiras diferencas sao estacionarias.### Ordem de diferenciacao (d):- d=0: serie original- d=1: primeiras diferencas (Y_t - Y_{t-1})- d=2: segundas diferencas ((Y_t - Y_{t-1}) - (Y_{t-1} - Y_{t-2}))

In [ ]:
def diferenciar(y, ordem=1):
    resultado = y.copy()
    for _ in range(ordem):
        resultado = np.diff(resultado)
    return resultado


### O que observar:1. Original tem correlacao alta (nao-estacionaria)2. Primeira diferenca reduz drasticamente a correlacao3. Segunda diferenca nao eh necessaria neste caso4. Diferenciacao remove a tendencia mas preserva mudancas locais### O que concluir:1. Diferenciacao eh transformacao reversivel2. d=1 frequentemente eh suficiente3. Diferenciacao eh essencial em modelos ARIMA4. Excesso de diferenciacao pode introduzir artefatos

## 5. Autocorrelacao (ACF)**ACF (AutoCorrelation Function)**: correlacao da serie com versoes atrasadas dela mesma.ACF(k) = correlacao(Y_t, Y_{t-k})### O que observar:1. ACF em lag 0 sempre igual a 1 (correlacao com si mesmo)2. ACF decai rapidamente para series estacionarias3. ACF decai lentamente para series com tendencia4. Picos periodicos em ACF indicam sazonalidade5. Intervalo confianca (linhas vermelhas) marca significancia6. ACF "corta" (salta para 0) em lag q para MA(q)7. ACF pode revelar ciclos nao-obvios visualmente8. ACF em lag 20+ identifica periodicidade de longo prazo### O que concluir:1. ACF rapido a zero => serie estacionaria2. ACF lento a zero => adicione diferenciacao3. ACF com picos periodicos => serie tem sazonalidade4. Amplitude de ACF decresce => dependencia temporal existe5. ACF multiplos picos => multiplos ciclos simultaneos### Interpretacao:- **ACF proxima a 1**: forte dependencia no lag k- **ACF proxima a 0**: fraca/nenhuma dependencia- **ACF decai lentamente**: serie nao-estacionaria ou tem tendencia- **ACF decai rapidamente**: serie estacionaria### Uso pratico:- Diagnosticar estacionariedade- Escolher lag em modelos AR (AutoRegressive)- Detectar sazonalidade (picos periodicos em ACF)

In [ ]:
def calcular_acf(y, nlags=40):
    y = y - np.mean(y)
    c0 = np.dot(y, y) / len(y)
    acf_vals = np.zeros(nlags + 1)
    acf_vals[0] = 1.0
    for lag in range(1, nlags + 1):
        c_lag = np.dot(y[:-lag], y[lag:]) / len(y)
        acf_vals[lag] = c_lag / c0
    return acf_vals


### O que observar:1. ACF ruido branco decai rapido (proximas a 0 apos lag 0)2. ACF com tendencia decai muito lentamente3. ACF sazonalidade tem picos periodicos4. Intervalos de confianca (linhas vermelhas) indicam significancia### O que concluir:1. ACF eh diagnostico visual poderoso2. Decaimento lento = nao-estacionaria3. Picos periodicos = sazonalidade4. ACF guia escolha de parametros em ARIMA

## 6. Autocorrelacao Parcial (PACF)**PACF (Partial AutoCorrelation Function)**: correlacao entre Y_t e Y_{t-k}, removendo efeito dos lags intermediarios.### O que observar:1. PACF lag 0 sempre igual a 12. PACF cai para 0 mais abruptamente que ACF3. PACF corta em lag p indica AR(p)4. Picos em PACF indicam AR componentes significantes5. Multiplos picos revelam dependencias em lags especificos6. PACF remove correlacoes indiretas (efeito "parcial")7. PACF vs ACF padroes diagnosticam tipo de modelo### O que concluir:1. PACF corta em lag p => usar AR(p)2. ACF corta mas PACF decai => usar MA3. Ambas decaem => usar ARMA (misto)4. Nenhuma corta claramente => serie precisa diferenciacao5. PACF ajuda escolher ordem de AR mais que ACF### Interpretacao:- **PACF proxima a 0 apos lag p**: usar AR(p)- **ACF proxima a 0 apos lag q**: usar MA(q)### Diferenca:- **ACF**: correlacao direta (pode incluir efeitos indiretos)- **PACF**: correlacao depois de remover correlacoes intermediarias### Uso em ARIMA:- PACF corta = AR(p)- ACF corta = MA(q)

In [ ]:
def calcular_pacf_aproximada(y, nlags=40):
    # Aproximacao simplificada usando regressao
    acf_vals = calcular_acf(y, nlags)
    pacf_vals = np.zeros(nlags + 1)
    pacf_vals[0] = 1.0
    pacf_vals[1] = acf_vals[1]
    for lag in range(2, nlags + 1):
        pacf_vals[lag] = acf_vals[lag]
    return pacf_vals


### O que observar:1. AR(1) tem ACF decrescente e PACF com pico em lag 12. MA(1) tem ACF com pico em lag 1 e PACF decrescente3. PACF "corta" (cai para 0) em modelos AR4. ACF "corta" em modelos MA### O que concluir:1. ACF e PACF diagnosticam ordem (p, q) de ARIMA2. PACF remove correlacoes indiretas3. Padroes visuais guiam escolha de modelo4. AR vs MA tem "assinaturas" diferentes em ACF/PACF

## 7. Janela Deslizante e Media Movel**Media Movel (Moving Average)**: suaviza serie calculando media em janelas.MA_t(k) = (Y_t + Y_{t-1} + ... + Y_{t-k+1}) / k### O que observar:1. Media movel simples (SMA) remove flutuacoes locais2. Janela maior suaviza mais mas introduz lag3. Media movel centrada usa dados futuros (invalida para predicao)4. Media movel exponencial (EMA) pondera recente mais5. EMA responde mais rapido a mudancas que SMA6. Tamanho janela k controla grau suavizacao7. SMA cria artefatos nas extremidades (efeito borda)### O que concluir:1. SMA maior => suavizacao maior mas lag maior2. EMA melhor para predicao real-time que SMA3. SMA centrada somente em analise retrospectiva4. Forward-looking MA valido para predicao5. Multiplas MAs indicam tendencia (cruzamento = sinal)### Tipos:- **Simples (SMA)**: media aritmetica- **Exponencial (EMA)**: pondera observacoes recentes mais- **Centrada**: usa observacoes antes e depois de t### Uso:- Suavizacao de ruido- Identificacao de tendencia- Baseline para comparacao

In [ ]:
def media_movel_simples(y, k):
    n = len(y)
    sma = np.zeros(n)
    for i in range(n):
        inicio = max(0, i - k//2)
        fim = min(n, i + k//2 + 1)
        sma[i] = np.mean(y[inicio:fim])
    return sma


### O que observar:1. SMA maior suaviza mais mas lag mais2. EMA responde mais rapido a mudancas recentes3. SMA eh simetrica, EMA eh assimetrica (favorece recente)4. Tradeoff entre suavizacao e responsividade### O que concluir:1. Media movel simples introduce lag2. Media movel exponencial eh mais responsiva3. Tamanho de janela k controla grau de suavizacao4. SMA bom para tendencia, EMA bom para predicicao real-time

## 8. Suavizacao Exponencial**Suavizacao Exponencial Simples (SES)**: versao de EMA focada em previsao.Y_hat_{t+1} = alpha * Y_t + (1 - alpha) * Y_hat_t### O que observar:1. Alpha controla peso dado observacao recente vs historico2. Alpha baixo => serie prevista eh lisa, segue lentamente mudancas3. Alpha alto => serie prevista eh reativa, rastreia flutuacoes4. Suavizacao dupla separa nivel de tendencia5. Previsto = nivel + tendencia torna extrapolacao possivel6. EMA pode ser adaptativa (alpha ajusta ao longo tempo)### O que concluir:1. Alpha ~ 0.3 eh compromise entre responsividade e suavizacao2. Alpha pequeno vira SMA ponderada (historico pesado)3. Suavizacao dupla captura duas dinamicas simultaneas4. Suavizacao eh mais simples que ARIMA mas menos flexivel5. SES util quando tendencia eh aproximadamente linear### Parametro alpha:- **alpha ~ 0**: muita inercia, suavizado demais- **alpha ~ 1**: reativo demais, segue ruido- **alpha ~ 0.3**: valor tipico bem-balanceado### Extensoes:- **Holt (SES + tendencia)**: Y_hat = nivel + tendencia- **Holt-Winters (SES + tendencia + sazonalidade)**: adiciona componente sazonalAqui implementamos versao simples.

In [ ]:
def suavizacao_exponencial(y, alpha=0.3):
    n = len(y)
    ses = np.zeros(n)
    ses[0] = y[0]
    for t in range(1, n):
        ses[t] = alpha * y[t] + (1 - alpha) * ses[t-1]
    return ses


### O que observar:1. Alpha baixo (0.1) segue serie original mas lisa2. Alpha alto (0.7) rastreia coluna por coluna3. Dupla separacao nivel+tendencia eh mais estruturada4. Previsto antecipa movimento baseado em tendencia### O que concluir:1. Alfa controla tradeoff entre suavizacao e responsividade2. Suavizacao dupla captura duas componentes separadas3. SES eh simples mas eficaz4. Adaptativo: pode ajustar alpha ao longo do tempo

## 9. Deteccao de Sazonalidade: FFT e Periodograma**FFT (Fast Fourier Transform)**: transforma serie do dominio tempo para dominio frequencia.### Ideia:- Componentes periodicas aparecem como picos em frequencias especificas- Frequencia alto = mudancas rapidas (ruido)- Frequencia baixo = mudancas lentas (tendencia)### Periodo = 1 / frequenciaSe pico em frequencia 0.03, periodo = 1/0.03 ~ 33 dias.### O que observar:1. Picos em FFT sao frequencias dominantes2. Amplitude do pico proporcional a forca da periodicidade3. Multiplos picos indicam multiplos ciclos4. Frequencia = 1/periodo (relacao inversa)### O que concluir:1. FFT revela ciclos nao-obvios visualmente2. Ciclos sazonais aparecem como multiplos picos3. Ruido distribui uniformemente (sem picos)4. FFT eh tecnica complementar ao ACF### Limitacoes:- FFT assume serie estacionaria- Nao detecta mudancas de sazonalidade no tempo- Melhor com series longas

In [ ]:
def detectar_sazonalidade_fft(y, sampling_rate=1):
    n = len(y)

    # Remover tendencia (diferenciar)
    y_detrended = np.diff(y)

    # FFT
    fft = np.fft.fft(y_detrended)
    potencia = np.abs(fft[:n//2])**2
    frequencias = np.fft.fftfreq(len(y_detrended), 1/sampling_rate)[:len(y_detrended)//2]

    # Top frequencias
    top_indices = np.argsort(potencia)[-5:][::-1]
    top_frequencias = frequencias[top_indices]
    top_periodos = 1 / (top_frequencias + 1e-10)

    return frequencias, potencia, top_periodos

# Gerar serie com sazonalidade clara
t = np.arange(365)
y_saz = 100 + 10*np.sin(2*np.pi*t/365) + 5*np.sin(2*np.pi*t/30) + np.random.normal(0, 1, 365)

freq, pot, periodos = detectar_sazonalidade_fft(y_saz, sampling_rate=1)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(y_saz, linewidth=1)
axes[0].set_title('Serie com Sazonalidade Anual e Mensal')
axes[0].set_ylabel('Valor')
axes[0].grid(True, alpha=0.3)

# Log scale para melhor visualizacao
axes[1].semilogy(freq, pot + 1e-10, linewidth=1)
axes[1].set_title('Espectro de Potencia (FFT)')
axes[1].set_xlabel('Frequencia')
axes[1].set_ylabel('Potencia (log)')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(0, 0.1)

plt.tight_layout()
plt.show()

print("Periodos detectados:")
for i, periodo in enumerate(periodos[:5]):
    if periodo > 0:
        print(f"  {i+1}. Periodo ~ {periodo:.1f} dias")

### O que observar:1. Picos em FFT correspondem a periodicidades2. Frequencia 1/365 ~ sazonalidade anual3. Frequencia 1/30 ~ sazonalidade mensal4. Escala log revela picos pequenos### O que concluir:1. FFT detecta multiplos ciclos simultaneos2. Frequencias altas = ruido aleatorio3. FFT pressupoe serie estacionaria4. Uteil para explorar dados, nao para predicao direta

## Pratica 3: Diferenciacao OtimaPara uma serie nao-estacionaria, determine a ordem de diferenciacao (d) que melhor reduz ACF.= None  # TAREFA DO ALUNO: Implemente aqui

In [ ]:
# PRATICA
resultado = None  # TAREFA DO ALUNO: Implemente aqui

## Pratica 2: ACF em Series DiferentesCrie 3 series (estacionaria, nao-estacionaria, sazonal). Calcule ACF para cada uma.= None  # TAREFA DO ALUNO: Implemente aqui

In [ ]:
# PRATICA
resultado = None  # TAREFA DO ALUNO: Implemente aqui

## Pratica 1: Teste sua CompreensaoGere uma serie temporal com componentes conhecidos. Decomponha e verifique se recupera os componentes.= None  # TAREFA DO ALUNO: Implemente aqui

In [ ]:
# PRATICA
resultado = None  # TAREFA DO ALUNO: Implemente aqui

## 10. Exercicios e Solucoes

Esta secao separa os enunciados das solucoes executaveis. Os exemplos usam apenas NumPy e Matplotlib para manter o notebook portavel.

### Exercicio 1: Decomposicao manual

Gere uma serie com tendencia linear, sazonalidade e ruido. Depois verifique se a soma dos componentes reconstrói a serie original.

In [ ]:
# SOLUCAO - Exercicio 1: decomposicao manual
np.random.seed(42)
n_pts = 200
t = np.arange(n_pts)
tendencia = 0.05 * t
sazonalidade = 3 * np.sin(2 * np.pi * t / 12)
ruido = np.random.randn(n_pts) * 0.5
y_exercicio = tendencia + sazonalidade + ruido

reconstruida = tendencia + sazonalidade + ruido
erro_maximo = np.max(np.abs(y_exercicio - reconstruida))

print(f'Erro maximo de reconstrucao: {erro_maximo:.8f}')
print(f'Tendencia final: {tendencia[-1]:.2f}')
print(f'Desvio padrao do ruido: {np.std(ruido):.2f}')

### Exercicio 2: ACF em series diferentes

Crie três séries, calcule a ACF dos primeiros 20 lags e classifique cada uma com uma regra simples.

In [ ]:
# SOLUCAO - Exercicio 2: ACF em series diferentes
np.random.seed(456)

y_ruido = np.random.normal(0, 1, 200)
y_tendencia = 50 + np.arange(200) * 0.3 + np.random.normal(0, 2, 200)
y_sazonal = 50 + 8 * np.sin(2 * np.pi * np.arange(200) / 30) + np.random.normal(0, 1, 200)

series = {
    'Ruido branco': y_ruido,
    'Tendencia': y_tendencia,
    'Sazonalidade': y_sazonal,
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (nome, serie) in zip(axes, series.items()):
    acf = calcular_acf(serie, nlags=20)
    estado = 'Estacionaria' if abs(acf[20]) < 0.2 else 'Nao-estacionaria'
    lags = np.arange(len(acf))
    ax.stem(lags, acf, basefmt=' ')
    ax.axhline(y=0.2, color='r', linestyle='--', alpha=0.5)
    ax.axhline(y=-0.2, color='r', linestyle='--', alpha=0.5)
    ax.set_title(f'{nome}\n{estado}')
    ax.set_xlabel('Lag')
    ax.set_ylabel('ACF')
    ax.grid(True, alpha=0.3)
    print(f'{nome}: ACF[20]={acf[20]:.3f} -> {estado}')

plt.tight_layout()
plt.show()

### Exercicio 3: Diferenciacao minima

Teste ordens de diferenciação e pare na menor ordem que reduz a autocorrelação de curto prazo.

In [ ]:
# SOLUCAO - Exercicio 3: diferenciacao minima
y_nao_est = y_tendencia

for d in range(4):
    y_d = np.diff(y_nao_est, n=d) if d > 0 else y_nao_est
    acf_d = calcular_acf(y_d, nlags=20)
    print(f'd={d}: ACF[1]={acf_d[1]:.3f}, variancia={np.var(y_d):.3f}')
    if abs(acf_d[1]) < 0.5:
        print(f'Ordem minima sugerida: d={d}')
        break

## 11. Erros Comuns

Os blocos abaixo mostram armadilhas frequentes em series temporais e a forma correta de tratar cada uma.

### Erro 1: regressao espuria em series nao-estacionarias

Series independentes podem parecer correlacionadas quando ambas têm passeio aleatório. Diferenciar ajuda a remover esse efeito.

In [ ]:
# Demonstracao: regressao espuria
np.random.seed(999)
t = np.arange(200)
y1 = np.cumsum(np.random.normal(0, 1, 200))
y2 = np.cumsum(np.random.normal(0, 1, 200))

coef_espurio = np.polyfit(y1, y2, 1)[0]
y1_est = np.diff(y1)
y2_est = np.diff(y2)
coef_correto = np.polyfit(y1_est, y2_est, 1)[0]

print(f'Coeficiente regressao em niveis: {coef_espurio:.4f}')
print(f'Coeficiente apos diferenciacao: {coef_correto:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y1, y2, alpha=0.5, s=20)
axes[0].plot(y1, np.polyval([coef_espurio, 0], y1), 'r-', label=f'y2={coef_espurio:.3f}*y1')
axes[0].set_title('Regressao espuria')
axes[0].set_xlabel('y1')
axes[0].set_ylabel('y2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].scatter(y1_est, y2_est, alpha=0.5, s=20, color='green')
axes[1].plot(y1_est, np.polyval([coef_correto, 0], y1_est), 'r-', label=f'y2={coef_correto:.3f}*y1')
axes[1].set_title('Apos diferenciacao')
axes[1].set_xlabel('Diff(y1)')
axes[1].set_ylabel('Diff(y2)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Erro 2: diferenciacao excessiva

Diferenciar alem do necessario aumenta variancia e remove sinal util.

In [ ]:
# Demonstracao: diferenciacao excessiva
y_original = 100 + 0.1 * np.arange(100) + 2 * np.sin(2 * np.pi * np.arange(100) / 30) + np.random.normal(0, 1, 100)
d0 = y_original
d1 = np.diff(d0)
d2 = np.diff(d1)
d3 = np.diff(d2)

print(f'Variancia d=0: {np.var(d0):.2f}')
print(f'Variancia d=1: {np.var(d1):.2f}')
print(f'Variancia d=2: {np.var(d2):.2f}')
print(f'Variancia d=3: {np.var(d3):.2f}')

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
for ax, data, label, color in zip(
    axes,
    [d0, d1, d2, d3],
    ['d=0 (Original)', 'd=1', 'd=2', 'd=3 (sobre-diferenciada)'],
    ['steelblue', 'orange', 'green', 'red'],
):
    ax.plot(np.arange(len(data)), data, 'o-', markersize=3, linewidth=1, color=color)
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Tempo')
plt.tight_layout()
plt.show()

### Erro 3: ignorar sazonalidade

Diferenciação comum remove tendência, mas não necessariamente remove ciclos sazonais.

In [ ]:
# Demonstracao: sazonalidade ignorada
t = np.arange(120)
y_saz = 100 + 10 * np.sin(2 * np.pi * t / 12) + np.random.normal(0, 1, 120)
y_diff_comum = np.diff(y_saz)
y_diff_sazonal = y_saz[12:] - y_saz[:-12]

acf_original = calcular_acf(y_saz, nlags=36)
acf_diff_comum = calcular_acf(y_diff_comum, nlags=35)
acf_diff_sazonal = calcular_acf(y_diff_sazonal, nlags=35)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for ax, acf, title in zip(
    axes,
    [acf_original, acf_diff_comum, acf_diff_sazonal],
    ['ACF original', 'ACF diferenca comum', 'ACF diferenca sazonal (lag=12)'],
):
    lags = np.arange(len(acf))
    ax.stem(lags, acf, basefmt=' ')
    ax.axvline(x=12, color='r', linestyle='--', alpha=0.5)
    ax.set_title(title)
    ax.set_ylabel('ACF')
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Lag')
plt.tight_layout()
plt.show()

### Erro 4: usar media movel centrada em predicao

Media movel centrada usa dados futuros. Para previsão, use apenas informação disponível no tempo da decisão.

In [ ]:
# Demonstracao: media movel centrada vaza futuro
y = 100 + 0.1 * np.arange(100) + 5 * np.sin(2 * np.pi * np.arange(100) / 30) + np.random.normal(0, 1, 100)

sma_centrada = np.zeros(100)
for i in range(100):
    inicio = max(0, i - 5)
    fim = min(100, i + 5 + 1)
    sma_centrada[i] = np.mean(y[inicio:fim])

sma_passada = np.zeros(100)
for i in range(100):
    inicio = max(0, i - 10)
    fim = i + 1
    sma_passada[i] = np.mean(y[inicio:fim])

ema = suavizacao_exponencial(y, alpha=0.2)
t = np.arange(100)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(t, y, 'o-', label='Original', alpha=0.5, markersize=3, linewidth=1)
axes[0].plot(t, sma_centrada, label='SMA centrada (invalida para predicao)', linewidth=2, linestyle='--')
axes[0].plot(t, sma_passada, label='SMA usando passado', linewidth=2)
axes[0].set_title('Media movel centrada vs passado')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, y, 'o-', label='Original', alpha=0.5, markersize=3, linewidth=1)
axes[1].plot(t, ema, label='EMA', linewidth=2)
axes[1].set_title('EMA para predicao')
axes[1].set_xlabel('Tempo')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Erro 5: confundir estacionariedade com ausencia de ciclos

Uma serie pode ter ciclos regulares e ainda manter propriedades estatisticas relativamente constantes.

In [ ]:
# Demonstracao: serie estacionaria com ciclos
t = np.arange(200)
y_ciclo = 5 * np.sin(2 * np.pi * t / 50) + 2 * np.sin(2 * np.pi * t / 17) + np.random.normal(0, 0.5, 200)
corr_ciclo = teste_estacionariedade_simples(y_ciclo)
acf_ciclo = calcular_acf(y_ciclo, nlags=50)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(t, y_ciclo, 'o-', markersize=3, linewidth=1)
axes[0].set_title(f'Serie com ciclos (corr lag-1 = {corr_ciclo:.3f})')
axes[0].set_ylabel('Valor')
axes[0].grid(True, alpha=0.3)

lags = np.arange(len(acf_ciclo))
axes[1].stem(lags, acf_ciclo, basefmt=' ')
axes[1].set_title('ACF: picos indicam ciclos regulares')
axes[1].set_ylabel('ACF')
axes[1].set_xlabel('Lag')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Correlacao lag-1: {corr_ciclo:.4f}')
print(f'Media: {np.mean(y_ciclo):.3f}')
print(f'Variancia: {np.var(y_ciclo):.3f}')

## 12. Resumo Executivo

**Series temporais** são sequencias ordenadas no tempo. Ordem, dependência temporal e vazamento de futuro são parte do problema, não detalhes.

**Componentes principais:** tendência, sazonalidade, ruído e ciclos.

**Ferramentas deste notebook:** decomposição, ACF/PACF, diferenciação, suavização e FFT.

**Conexão com próximos módulos:** ARIMA/SARIMA usam estacionariedade e ACF/PACF; modelos de ML e deep learning usam features temporais e janelas sequenciais.

### Referencia rapida

```python
np.diff(y, n=d)                  # diferenciacao
np.convolve(y, np.ones(k)/k)     # media movel simples
np.fft.fft(y)                    # transformada de Fourier
np.correlate(y, y, mode='full')  # autocorrelacao bruta
```